# 03 · Evaluation

*Code companion to an MSc dissertation on score-based generative
modelling of financial time series (Queen Mary University of London,
2026) — see the repository README for the reference.*

This notebook evaluates generated samples against real data along two
complementary axes: the stylized facts of asset returns and
distributional similarity. All evaluations share one calibration design:

- the real data are split once into **disjoint, seed-fixed chunks**;
- every statistic is computed per chunk, giving a **real-vs-real floor** with error bars;
- **noise ceilings** (moment-matched Gaussian, uniform) bound the scale from above;
- a model is reported by its position between floor and ceiling as a **z-score against the floor**.

| Section | Evaluation | What it measures |
|---|---|---|
| 1 | Stylized facts | Linear unpredictability, heavy tails (excess kurtosis), volatility clustering, leverage effect |
| 2 | Distribution plots | Marginal density, log-density, QQ and tail survival; per-window statistics |
| 3 | C2ST | Classifier two-sample test with a GRU discriminator |
| 4 | Sliced Wasserstein | Projection-averaged one-dimensional Wasserstein distance (SW$_1$ reported) |
| 5 | MMD | Unbiased maximum mean discrepancy with a multi-scale Gaussian kernel |

**Naming convention for generated samples** (`gen_data/`):
`gs_<sampler>_conf<k>_s<stride>_noise<sigma_max>_seed<seed>.pt`

## 0 · Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import torch

from config import data_config, model_config
from data import get_data, DailyLogReturnsData
from evaluation import (
    make_eval_chunks, build_real_benchmark, evaluate_stylized_facts,
    plot_model_comparison_curves, c2st_battery, swd_battery, mmd_battery,
)
from plots import plot_marginals, plot_window_stats

device = model_config["DEVICE"]

## 0.1 · Real data

The evaluation pool is rebuilt at `stride=1` so the benchmark uses every
available window; the stride used to *train* a model is recorded in that
model's run name.

In [ ]:
snp_data = get_data({
    "ticker": data_config["TICKER"],
    "start":  data_config["START"],
    "end":    data_config["END"],
})
dataset = DailyLogReturnsData(
    log_returns=snp_data["log_return"],
    window_size=data_config["WINDOW_SIZE"],
    stride=1,
    normalize=True,
)
X_train = torch.stack([dataset[i].squeeze(0) for i in range(len(dataset))])
print(f"Real windows: {tuple(X_train.shape)}   pooled std = {X_train.std():.4f}")

## 0.2 · Generated data

Point `GEN_PATH` at the sample set to evaluate. `MODEL_LABEL` is used in
tables and figure legends.

In [ ]:
GEN_PATH    = "gen_data/gs_PC_conf1_s20_noise10_seed0.pt"
MODEL_LABEL = "Conf1 D3 (PC)"   # Conf<k> D<dataset> (<sampler>)

generated_samples = torch.load(GEN_PATH, map_location="cpu")
print(f"Generated: {tuple(generated_samples.shape)}   "
      f"mean = {generated_samples.mean():.4f}   std = {generated_samples.std():.4f}")

## 0.3 · Shared evaluation chunks

Built **once** and reused by every evaluation below. Chunks are disjoint,
sized to the generated set, and deterministic in `seed`, so all metrics
describe exactly the same real data.

In [ ]:
chunks, info = make_eval_chunks(X_train, n_per_set=len(generated_samples), seed=42)

## 1 · Stylized facts

### 1.1 · Real benchmark
Computed once per chunk size and cached to `benchmarks/`; subsequent calls
load the cache so every model is compared against identical reference values.

In [ ]:
bench, real_table = build_real_benchmark(
    chunks, lags=(1, 10, 20),
    save_path=f"benchmarks/real_L{info['window_length']}_n{info['n_per_set']}.json",
)

### 1.2 · Generated vs real

In [ ]:
sf_results = evaluate_stylized_facts(MODEL_LABEL, generated_samples, chunks, bench)

### 1.3 · Comparing several models

Any number of sample sets can be overlaid on the real band, e.g. to compare
the effect of model capacity, sampling method or training data volume. Keys are the legend labels, in the form `Conf<k> D<d> (<sampler>)`.
`panels` selects `"acf"`, `"acf_abs"`, `"lev"` in any order; `title=None`
suppresses the figure title when a caption will carry it.

In [ ]:
models = {
    "Conf1 D3 (PC)": torch.load("gen_data/gs_PC_conf1_s20_noise10_seed0.pt", map_location="cpu"),
    "Conf2 D3 (PC)": torch.load("gen_data/gs_PC_conf2_s20_noise10_seed0.pt", map_location="cpu"),
}

plot_model_comparison_curves(
    chunks, models, max_lag=50, mark_lags=(10, 20),
    panels=("acf", "acf_abs", "lev"), title=None,
    save_path="figures/comparison_D3_PC_curves.png",
)

## 2 · Distribution plots

`plot_marginals` compares pooled values (density, log-density, QQ, tail
survival); `standardize_tails=True` z-scores each series before the QQ and
survival panels so tail *shape* is compared free of scale mismatch.
`plot_window_stats` compares the distribution of per-window statistics.

In [ ]:
plot_marginals(X_train, models, standardize_tails=True, title=None,
               save_path="figures/comparison_D3_PC_marginals.png")

plot_window_stats(chunks, models, show_means=False, title=None,
                  save_path="figures/comparison_D3_PC_window_stats.png")

## 3 · Classifier two-sample test (C2ST)

A two-layer GRU discriminator (hidden size 64) is trained per comparison with
validation-based early stopping and evaluated once on the held-out test split
(64/16/20 stratified split, classes balanced), over `n_seeds` seeds. Accuracy near 0.5 indicates high similarity. The real-vs-real
row establishes the floor of the procedure and the real-vs-Gaussian row
confirms that the classifier has the capacity to separate distributions.

In [ ]:
c2st_results = c2st_battery(
    chunks, generated_samples.cpu(),
    model_name=MODEL_LABEL.replace(" ", "_"),
    device=device, n_seeds=3, num_layers=2,
    checkpoint_dir="./checkpoints/classifier",
)

## 4 · Sliced Wasserstein distance

$\mathrm{SW}_p = \big(\mathbb{E}_\theta\, W_p^p\big)^{1/p}$ over random unit
directions, with the one-dimensional $W_p$ computed from the sorted
projections. The finite-sample estimate
is biased in the sample size, so every comparison is between equal-sized sets.

In [ ]:
swd_results = swd_battery(chunks, generated_samples.cpu(),
                          device=device, projections=2000, seed=42)

## 5 · Maximum mean discrepancy

Unbiased MMD$^2$ estimate with a multi-scale Gaussian kernel whose
bandwidths are set once from the pooled real data by the median heuristic.
The estimator is unbiased at any sample size, so the full
generated set is compared against each chunk. Values are reported unclamped:
a slightly negative real-vs-real value is expected under the null.

In [ ]:
mmd_results = mmd_battery(chunks, generated_samples.cpu(), seed=42)